# Loan Default Prediction Project

**Objective:** Build a machine learning model to predict whether a loan applicant is likely to default or repay the loan based on historical data.

## Phase 1 - Data Preprocessing
First, we will import necessary libraries, load the data, and handle missing values, drop unnecessary columns, and encode categorical data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset
df = pd.read_csv('loan_data.csv')

# Remove index column if present
if df.columns[0] == 'Unnamed: 0':
    df = df.drop(columns=[df.columns[0]])

df.head()

In [ ]:
# Drop 'Loan_ID' as it's not useful for prediction
df = df.drop(columns=['Loan_ID'])

# Handle missing values
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

print("Missing values after preprocessing:")
print(df.isnull().sum())

## Phase 2 - Exploratory Data Analysis (EDA)
Visualizing distributions and correlations between variables.

In [ ]:
# Visualize distributions of Loan Amount and Applicant Income
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['LoanAmount'], kde=True, bins=30)
plt.title('Loan Amount Distribution')

plt.subplot(1, 2, 2)
sns.histplot(df['ApplicantIncome'], kde=True, bins=30)
plt.title('Applicant Income Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Compare Features vs Loan_Status
plt.figure(figsize=(10, 5))
sns.countplot(x='Credit_History', hue='Loan_Status', data=df)
plt.title('Loan Status by Credit History')
plt.show()

In [ ]:
# Convert Categorical data using Label Encoding
le = LabelEncoder()
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Matrix')
plt.show()

## Phase 3 - Model Building
We will split the data into 80-20 train-test sets, scale the features, and train 3 models: Logistic Regression, Decision Tree, and Random Forest.

In [ ]:
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# Evaluate models
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\n")

## Phase 4 - Model Comparison
Based on the results, **Logistic Regression** is the best performing model with an accuracy of ~82.8% and the highest F1-score (~0.88).
It handles the linear relationship well, especially considering `Credit_History` is the strongest predictor for Loan Status.